In [1]:
import pandas as pd
import numpy as np

cohort = pd.read_pickle(
    "../results/cohort_features_checkpoint.pkl"
)

# All categorical predictors from the verified paper map
all_categorical_cols = [
    "RIAGENDR",          # Gender
    "RIDRETH1",          # Race
    "INDHHINC",          # Income
    "CVDFITLV",          # Cardiovascular fitness level
    "PAQ180",            # Physical activity
    "BPQ020",            # Hypertension history
    "XY_Estimation",     # DNAm estimated sex
    "MCQ160B",           # Heart failure history
    "MCQ160C",           # Coronary artery disease history
    "MCQ160F",           # Cerebrovascular/stroke history
    "DERIVED_HL_MED",    # Hyperlipidemia medication
    "DERIVED_HTN_MED",   # Hypertension medication
    "DERIVED_DM_MED",    # Diabetes medication
    "DIQ010",            # Diabetes history
]

# Only use columns that currently exist
categorical_cols = [
    col for col in all_categorical_cols
    if col in cohort.columns
]

missing_categorical = [
    col for col in all_categorical_cols
    if col not in cohort.columns
]

print("Categorical columns available now:")
print(categorical_cols)

print("\nStill need to build later:")
print(missing_categorical)

Categorical columns available now:
['RIAGENDR', 'RIDRETH1', 'INDHHINC', 'CVDFITLV', 'PAQ180', 'BPQ020', 'MCQ160B', 'MCQ160C', 'MCQ160F', 'DIQ010']

Still need to build later:
['XY_Estimation', 'DERIVED_HL_MED', 'DERIVED_HTN_MED', 'DERIVED_DM_MED']


In [2]:
cohort_encoded = pd.get_dummies(
    cohort,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 303)


In [3]:
for col in categorical_cols:
    new_cols = [
        c for c in cohort_encoded.columns
        if c.startswith(col + "_")
    ]

    print(col, "->", new_cols)

RIAGENDR -> ['RIAGENDR_1.0', 'RIAGENDR_2.0']
RIDRETH1 -> ['RIDRETH1_1.0', 'RIDRETH1_2.0', 'RIDRETH1_3.0', 'RIDRETH1_4.0', 'RIDRETH1_5.0']
INDHHINC -> ['INDHHINC_1.0', 'INDHHINC_2.0', 'INDHHINC_3.0', 'INDHHINC_4.0', 'INDHHINC_5.0', 'INDHHINC_6.0', 'INDHHINC_7.0', 'INDHHINC_8.0', 'INDHHINC_9.0', 'INDHHINC_10.0', 'INDHHINC_11.0', 'INDHHINC_12.0', 'INDHHINC_13.0', 'INDHHINC_77.0', 'INDHHINC_99.0']
CVDFITLV -> ['CVDFITLV_1.0', 'CVDFITLV_2.0', 'CVDFITLV_3.0']
PAQ180 -> ['PAQ180_1.0', 'PAQ180_2.0', 'PAQ180_3.0', 'PAQ180_4.0']
BPQ020 -> ['BPQ020_1.0', 'BPQ020_2.0', 'BPQ020_9.0']
MCQ160B -> ['MCQ160B_1.0', 'MCQ160B_2.0', 'MCQ160B_7.0', 'MCQ160B_9.0']
MCQ160C -> ['MCQ160C_1.0', 'MCQ160C_2.0', 'MCQ160C_7.0', 'MCQ160C_9.0']
MCQ160F -> ['MCQ160F_1.0', 'MCQ160F_2.0', 'MCQ160F_9.0']
DIQ010 -> ['DIQ010_1.0', 'DIQ010_2.0', 'DIQ010_3.0', 'DIQ010_7.0', 'DIQ010_9.0']


In [7]:
cohort_ohe = cohort.copy()

special_missing = {
    "INDHHINC": [77, 99],
    "PAQ180": [7, 9],
    "BPQ020": [7, 9],
    "MCQ160B": [7, 9],
    "MCQ160C": [7, 9],
    "MCQ160F": [7, 9],
    "DIQ010": [7, 9],
}

for col, codes in special_missing.items():
    if col in cohort_ohe.columns:
        cohort_ohe[col] = cohort_ohe[col].replace(codes, np.nan)

In [5]:
cohort_encoded = pd.get_dummies(
    cohort_ohe,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort_ohe.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 293)


In [6]:
bad_dummy_cols = [
    c for c in cohort_encoded.columns
    if any(
        c.endswith(suffix)
        for suffix in [
            "_7.0",
            "_9.0",
            "_77.0",
            "_99.0"
        ]
    )
]

print("Invalid dummy columns:", bad_dummy_cols)

Invalid dummy columns: ['INDHHINC_7.0', 'INDHHINC_9.0']
